# 🌀 RayCash-Pro — V4 Ensemble (3 architectures)

Entraîne **3 modèles distincts** sur le même dataset (TrashNet + Drinking Waste capé) pour les utiliser en ensemble côté serveur.

**Architectures choisies** (volontairement différentes pour maximiser la diversité) :
1. **EfficientNetV2-B0** — bon généraliste, ~7M params
2. **MobileNetV2** — léger, bon sur les contours, ~3.5M params
3. **ResNet50** — plus profond, bon sur les textures, ~25M params

Plus les architectures sont **différentes**, plus l'ensemble est efficace (decorrélation des erreurs).

**Output** : 3 fichiers `model_<archi>.tflite` + 1 `labels.txt`. Tu les places dans `raycash-pro/server/models/ensemble/`.

**Durée** : ~30-45 min sur T4 GPU (les 3 modèles entraînés en séquence).

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU :', gpus)
assert gpus, 'Active le GPU dans Exécution → Modifier le type d\'exécution'

In [ ]:
import os, shutil, random, pathlib, zipfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

WORK_DIR = pathlib.Path('/content/raycash')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

## 1️⃣ Datasets (TrashNet + Drinking Waste capé)

Identique à V3.

In [ ]:
TRASHNET_URL = 'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip'
TRASHNET_ZIP = WORK_DIR / 'trashnet.zip'
TRASHNET_DIR = WORK_DIR / 'dataset-resized'

if not TRASHNET_DIR.exists():
    print('Téléchargement TrashNet...')
    urllib.request.urlretrieve(TRASHNET_URL, TRASHNET_ZIP)
    with zipfile.ZipFile(TRASHNET_ZIP) as z:
        z.extractall(WORK_DIR)

for d in sorted(TRASHNET_DIR.iterdir()):
    if d.is_dir():
        print(f'  {d.name}: {len(list(d.glob("*.jpg")))} images')

In [ ]:
# Dataset secondaire optionnel via Kaggle (voir instructions notebook V3)
WARP_DIR = WORK_DIR / 'warp'
WARP_ENABLED = False

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    !pip install -q kaggle
    if not WARP_DIR.exists():
        WARP_DIR.mkdir()
        !kaggle datasets download -d arkadiyhacks/drinking-waste-classification -p {WARP_DIR} --unzip
    WARP_ENABLED = True
    print('✅ Dataset secondaire chargé')
except Exception as e:
    print(f'⚠️ Skip dataset secondaire ({type(e).__name__}: {e})')
    WARP_ENABLED = False

## 2️⃣ Fusion + split stratifié + rebalance

In [ ]:
RAYCASH_CLASSES = ['Aluminium', 'Carton', 'Inconnu', 'Papier', 'Plastique', 'Verre']
TRASHNET_MAP = {'cardboard':'Carton','glass':'Verre','metal':'Aluminium','paper':'Papier','plastic':'Plastique','trash':'Inconnu'}
DRINKING_WASTE_MAP = {
    'Aluminium Cans':'Aluminium','Aluminum_Cans':'Aluminium','aluminum cans':'Aluminium',
    'Glass':'Verre','Glass_Bottles':'Verre',
    'Plastic':'Plastique','Plastic_Bottles':'Plastique',
    'Paper Carton':'Carton','Paper_Cartons':'Carton',
}
MAX_FROM_SECONDARY = {'Aluminium':400, 'Verre':400, 'Plastique':400, 'Carton':400}

SPLIT_DIR = WORK_DIR / 'split'
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
TRAIN_RATIO, VAL_RATIO = 0.8, 0.1

def collect_images(root, mapping):
    bucket = {}
    if not root.exists(): return bucket
    for d in root.rglob('*'):
        if not d.is_dir(): continue
        target = next((v for k,v in mapping.items() if d.name.lower()==k.lower()), None)
        if target is None: continue
        imgs = list(d.glob('*.jpg'))+list(d.glob('*.jpeg'))+list(d.glob('*.png'))
        if imgs: bucket.setdefault(target, []).extend(imgs)
    return bucket

buckets = {c: [] for c in RAYCASH_CLASSES}
for cls, paths in collect_images(TRASHNET_DIR, TRASHNET_MAP).items():
    buckets[cls].extend(paths)
if WARP_ENABLED:
    for cls, paths in collect_images(WARP_DIR, DRINKING_WASTE_MAP).items():
        cap = MAX_FROM_SECONDARY.get(cls, len(paths))
        random.Random(SEED).shuffle(paths)
        buckets[cls].extend(paths[:cap])

for cls, files in buckets.items():
    random.Random(SEED).shuffle(files)
    n = len(files); n_train = int(n*TRAIN_RATIO); n_val = int(n*VAL_RATIO)
    splits = {'train':files[:n_train], 'val':files[n_train:n_train+n_val], 'test':files[n_train+n_val:]}
    for split_name, items in splits.items():
        dest = SPLIT_DIR / split_name / cls
        dest.mkdir(parents=True, exist_ok=True)
        for f in items:
            shutil.copy(f, dest / (f.parent.parent.name+'_'+f.parent.name+'_'+f.name))

for split in ['train','val','test']:
    total = sum(len(list((SPLIT_DIR/split/cls).glob('*'))) for cls in RAYCASH_CLASSES)
    print(f'[{split}] {total}')

## 3️⃣ Pipelines tf.data (sans mixup pour ce notebook — chaque modèle apprend la même chose)

On garde l'augmentation forte, mais on retire le mixup pour simplifier le code multi-modèles. Mixup peut être ré-ajouté facilement plus tard.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = len(RAYCASH_CLASSES)

def make_raw(split):
    return tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split, image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_names=RAYCASH_CLASSES,
        shuffle=(split=='train'), seed=SEED,
    )

train_raw = make_raw('train')
val_raw   = make_raw('val')
test_raw  = make_raw('test')

data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(lambda x,y: (data_augment(x, training=True), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_raw.prefetch(AUTOTUNE)
test_ds  = test_raw.prefetch(AUTOTUNE)

# Class weights pour rééquilibrer les classes faibles (Aluminium, Inconnu)
y_train = np.concatenate([y for _, y in train_raw], axis=0)
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight_dict = {i: float(w) for i, w in enumerate(weights)}
print('Class weights :', class_weight_dict)

## 4️⃣ Factory : fonction qui construit n'importe lequel des 3 modèles

Pour réutilisable : `build_model('efficientnet')`, `build_model('mobilenet')`, `build_model('resnet50')`.

In [ ]:
BACKBONES = {
    'efficientnet': {
        'class': tf.keras.applications.EfficientNetV2B0,
        'kwargs': dict(include_preprocessing=True),
        'preprocess': lambda x: x,  # déjà inclus
    },
    'mobilenet': {
        'class': tf.keras.applications.MobileNetV2,
        'kwargs': {},
        'preprocess': tf.keras.applications.mobilenet_v2.preprocess_input,
    },
    'resnet50': {
        'class': tf.keras.applications.ResNet50,
        'kwargs': {},
        'preprocess': tf.keras.applications.resnet50.preprocess_input,
    },
}

L2_REG = 1e-4
LABEL_SMOOTHING = 0.1
DROPOUT = 0.4

def build_model(name: str):
    cfg = BACKBONES[name]
    base = cfg['class'](
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        **cfg['kwargs'],
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = cfg['preprocess'](inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    outputs = tf.keras.layers.Dense(
        NUM_CLASSES, activation='softmax',
        kernel_regularizer=tf.keras.regularizers.l2(L2_REG),
    )(x)
    return tf.keras.Model(inputs, outputs), base


def train_one(name: str, epochs_head=15, epochs_ft=10):
    print(f'\n{"="*60}\nEntraînement : {name}\n{"="*60}')
    model, base = build_model(name)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING) \
        if hasattr(tf.keras.losses.SparseCategoricalCrossentropy.__init__, 'label_smoothing') \
        else tf.keras.losses.SparseCategoricalCrossentropy()
    # Note : label_smoothing exige one-hot pour CategoricalCE ; SparseCE ne le supporte pas directement.
    # On utilise SparseCE pour simplicité (label smoothing désactivé ici, compensé par dropout fort).
    es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)

    # Phase 1 — head only
    cosine_head = tf.keras.optimizers.schedules.CosineDecay(1e-3, epochs_head * len(train_raw))
    model.compile(optimizer=tf.keras.optimizers.Adam(cosine_head), loss=loss_fn, metrics=['accuracy'])
    h_head = model.fit(train_ds, validation_data=val_ds, epochs=epochs_head, callbacks=[es], class_weight=class_weight_dict, verbose=2)

    # Phase 2 — fine-tune 20 dernières couches
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    cosine_ft = tf.keras.optimizers.schedules.CosineDecay(5e-5, epochs_ft * len(train_raw))
    model.compile(optimizer=tf.keras.optimizers.Adam(cosine_ft), loss=loss_fn, metrics=['accuracy'])
    h_ft = model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft, callbacks=[es], class_weight=class_weight_dict, verbose=2)

    return model, h_head, h_ft

## 5️⃣ Entraîner les 3 modèles

Chaque modèle est entraîné séquentiellement (un seul GPU partagé). Durée ~10-15 min par modèle.

In [ ]:
models_trained = {}
histories = {}

for arch in ['efficientnet', 'mobilenet', 'resnet50']:
    tf.keras.backend.clear_session()  # libère la mémoire GPU entre les runs
    model, h_head, h_ft = train_one(arch)
    models_trained[arch] = model
    histories[arch] = (h_head, h_ft)

## 6️⃣ Évaluation individuelle + ensemble

In [ ]:
y_true = []
for _, batch_y in test_raw:
    y_true.extend(batch_y.numpy())
y_true = np.array(y_true)

# Prédictions individuelles
individual_preds = {}
for arch, model in models_trained.items():
    probs = model.predict(test_raw, verbose=0)
    individual_preds[arch] = probs
    acc = (np.argmax(probs, axis=1) == y_true).mean()
    print(f'{arch:15} test acc = {acc*100:.2f}%')

# Ensemble = moyenne des probabilités
ensemble_probs = np.mean(list(individual_preds.values()), axis=0)
ensemble_pred = np.argmax(ensemble_probs, axis=1)
ensemble_acc = (ensemble_pred == y_true).mean()
print(f'{"ENSEMBLE":15} test acc = {ensemble_acc*100:.2f}%')

print('\n--- Classification report (ensemble) ---')
print(classification_report(y_true, ensemble_pred, target_names=RAYCASH_CLASSES))

cm = confusion_matrix(y_true, ensemble_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=RAYCASH_CLASSES, yticklabels=RAYCASH_CLASSES)
plt.xlabel('Prédit'); plt.ylabel('Vérité')
plt.title(f'Confusion matrix — Ensemble V4 ({ensemble_acc*100:.1f}%)')
plt.tight_layout(); plt.show()

## 7️⃣ Analyse uncertainty

On mesure 2 signaux d'incertitude par prédiction :
1. **Max prob** : confiance de la classe top-1. < 0.65 → flou
2. **Entropie** : dispersion de la distribution. > 1.0 (sur 6 classes) → flou
3. **Désaccord ensemble** : variance entre les 3 modèles. > 0.15 → flou

Avec ces signaux, on peut rejeter les cas ambigus à l'inférence.

In [ ]:
def entropy(p, eps=1e-10):
    return -np.sum(p * np.log(p + eps), axis=-1)

max_probs   = np.max(ensemble_probs, axis=1)
entropies   = entropy(ensemble_probs)
# Désaccord = écart-type des prédictions argmax entre modèles (proxy simple)
stacked = np.stack(list(individual_preds.values()))  # [3, N, 6]
disagreement = np.std(stacked, axis=0).max(axis=-1)  # [N]

# Évaluons l'effet d'un seuil de confiance sur l'accuracy/coverage
print(f'{"Threshold":12} {"Accept rate":15} {"Acc sur acceptes":20}')
for thr in [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
    mask = max_probs >= thr
    coverage = mask.mean()
    acc_on_accepted = (ensemble_pred[mask] == y_true[mask]).mean() if mask.any() else 0
    print(f'  {thr:5.2f}       {coverage*100:6.1f}%         {acc_on_accepted*100:6.2f}%')

print('\n→ Choisis le seuil qui donne >97% acc tout en gardant >70% accept rate')
print('→ Côté serveur, on rejettera les cas en dessous de ce seuil avec un message UI clair.')

## 8️⃣ Export TFLite des 3 modèles + labels

In [ ]:
def representative_dataset():
    for images, _ in train_raw.take(100 // BATCH_SIZE + 1):
        for img in images:
            yield [tf.cast(tf.expand_dims(img, 0), tf.float32)]

OUT_DIR = WORK_DIR / 'ensemble_export'
OUT_DIR.mkdir(exist_ok=True)

for arch, model in models_trained.items():
    print(f'Conversion TFLite : {arch}...')
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    tflite_bytes = converter.convert()
    path = OUT_DIR / f'model_{arch}.tflite'
    path.write_bytes(tflite_bytes)
    print(f'  → {path.name} ({path.stat().st_size / 1024:.0f} KB)')

# labels.txt identique pour tous
(OUT_DIR / 'labels.txt').write_text('\n'.join(f'{i} {n}' for i, n in enumerate(RAYCASH_CLASSES)))
print(f'\nFichiers dans {OUT_DIR}:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')

## 9️⃣ Zipper + télécharger tout

In [ ]:
zip_path = WORK_DIR / 'ensemble_v4.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in OUT_DIR.iterdir():
        z.write(f, arcname=f.name)
print(f'ZIP : {zip_path.name} ({zip_path.stat().st_size / 1024:.0f} KB)')

from google.colab import files
files.download(str(zip_path))